# 1.Imports, Widgets, and Configuration

In [0]:
from datetime import datetime, timezone
from typing import List

from pyspark.sql import functions as F
from pyspark.sql import types as T

# 1. Define Widgets
dbutils.widgets.text("catalog", "enterprise")
dbutils.widgets.text("obs_schema", "gold_observability")
dbutils.widgets.text("obs_table", "pipeline_metrics_daily")
dbutils.widgets.text("pipeline_name", "market_data_pipeline")
dbutils.widgets.text("freshness_warn_minutes", "30")
dbutils.widgets.text("freshness_fail_minutes", "180")

# 2. Parse Parameters
CATALOG = dbutils.widgets.get("catalog").strip()
OBS_SCHEMA = dbutils.widgets.get("obs_schema").strip()
OBS_TABLE = dbutils.widgets.get("obs_table").strip()
PIPELINE = dbutils.widgets.get("pipeline_name").strip()
WARN_MIN = float(dbutils.widgets.get("freshness_warn_minutes"))
FAIL_MIN = float(dbutils.widgets.get("freshness_fail_minutes"))

OBS_TBL = f"{CATALOG}.{OBS_SCHEMA}.{OBS_TABLE}"

# 3. Global Time Context
now_utc = datetime.now(timezone.utc)
metric_date = now_utc.date()

print(f"[INFO] Observability target: {OBS_TBL}")
print(f"[INFO] pipeline={PIPELINE}, metric_date={metric_date}, now_utc={now_utc.isoformat()}")

# 2.Define Monitoring Targets (Metadata)
# event_ts_col: Column for freshness
# key_cols: Columns for distinct check
# null_cols: Columns for null rate check

In [0]:
# Day3.2 module bootstrap (platform-observability)
import sys
from pathlib import Path

_local_src = str(Path.cwd() / "src")
if _local_src not in sys.path:
    sys.path.insert(0, _local_src)

from lakehouse.observability_rules import (
    freshness_minutes as _freshness_minutes_shared,
)
from lakehouse.observability_rules import (
    status_from_freshness as _status_from_freshness_shared,
)

print("[day3.2] imported shared observability rules")

In [0]:
targets = [
    {
        "table": f"{CATALOG}.bronze_market.crypto_ohlc_raw",
        "event_ts_col": "ingestion_ts",
        "key_cols": ["source", "symbol", "interval", "event_time", "run_id"],
        "null_cols": ["source", "symbol", "interval", "raw_json", "ingestion_ts"],
        "warn_min": 60,
        "fail_min": 360,
    },
    {
        "table": f"{CATALOG}.silver_market.crypto_ohlc_1d",
        "event_ts_col": "bar_end_ts",
        "key_cols": ["source", "symbol", "bar_start_ts"],
        "null_cols": [
            "source",
            "symbol",
            "bar_start_ts",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "p_date",
        ],
        "warn_min": 1440,
        "fail_min": 4320,
    },
    {
        "table": f"{CATALOG}.silver_ref.ecb_fx_daily",
        "event_ts_col": "rate_date",
        "key_cols": ["rate_date", "base_ccy", "quote_ccy"],
        "null_cols": ["rate_date", "base_ccy", "quote_ccy", "fx_rate", "ingestion_ts"],
        "warn_min": 2880,
        "fail_min": 10080,
    },
    {
        "table": f"{CATALOG}.silver_ref.fred_series_daily",
        "event_ts_col": "obs_date",
        "key_cols": ["series_id", "obs_date"],
        "null_cols": ["series_id", "obs_date", "value", "ingestion_ts"],
        "warn_min": 10080,
        "fail_min": 20160,
    },
    {
        "table": f"{CATALOG}.gold_market.ohlc_1d",
        "event_ts_col": "bar_end_ts",
        "key_cols": ["source", "symbol", "bar_start_ts"],
        "null_cols": [
            "source",
            "symbol",
            "bar_start_ts",
            "open",
            "high",
            "low",
            "close",
            "volume",
            "p_date",
        ],
        "warn_min": 1440,
        "fail_min": 4320,
    },
    {
        "table": f"{CATALOG}.gold_ref.market_macro_daily",
        "event_ts_col": "mart_ts",
        "key_cols": ["source", "symbol", "trade_date"],
        "null_cols": [
            "source",
            "symbol",
            "trade_date",
            "close_px",
            "daily_volume",
            "eurusd_rate",
            "fedfunds",
            "mart_ts",
        ],
        "warn_min": 1440,
        "fail_min": 4320,
    },
]


# 3. Helper Functions Library

In [0]:
def safe_table_exists(full_name: str) -> bool:
    try:
        spark.table(full_name).limit(1).count()
        return True
    except Exception:
        return False


def compute_null_rate(df, cols: List[str]) -> float:
    # fraction of NULL across all selected cols (averaged)
    total = df.count()
    if total == 0:
        return 0.0
    exprs = [
        (F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)) / F.lit(total)).alias(c) for c in cols
    ]
    row = df.select(exprs).collect()[0].asDict()
    return float(sum(row.values()) / len(row)) if cols else 0.0


def compute_duplicate_rate(df, key_cols: List[str]) -> float:
    total = df.count()
    if total == 0 or not key_cols:
        return 0.0
    distinct_cnt = df.select([F.col(c) for c in key_cols]).distinct().count()
    return float((total - distinct_cnt) / total)


def compute_max_event_ts(df, event_ts_col: str):
    if event_ts_col not in df.columns:
        return None
    row = df.select(F.max(F.col(event_ts_col).cast("timestamp")).alias("mx")).collect()[0]
    return row["mx"]


def freshness_minutes(max_ts) -> float:
    fm = float(_freshness_minutes_shared(max_ts, now_utc=now_utc))
    if fm == float("inf"):
        return fm
    # cap negative freshness caused by date->timestamp at 00:00 and timezone edges
    return max(0.0, fm)


def status_from_freshness(fm: float, warn_min: float, fail_min: float) -> str:
    return _status_from_freshness_shared(fm, warn_min=warn_min, fail_min=fail_min)

# 4. Infrastructure: Ensure Target Table Exists

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {OBS_TBL} (
  metric_date         DATE,
  pipeline            STRING,
  table_name          STRING,
  row_count           BIGINT,
  max_event_ts        TIMESTAMP,
  freshness_minutes   DOUBLE,
  null_rate           DOUBLE,
  duplicate_rate      DOUBLE,
  status              STRING,
  computed_ts         TIMESTAMP
)
USING DELTA
PARTITIONED BY (metric_date)
""")

# 5.Main Logic: Iterate and Compute Metrics

In [0]:
rows = []
computed_ts = datetime.now(timezone.utc)

for t in targets:
    tbl = t["table"]
    print(f"[INFO] Measuring: {tbl}")

    if not safe_table_exists(tbl):
        print(f"[WARN] Table not found or unreadable: {tbl} -> writing FAIL row")
        rows.append(
            (metric_date, PIPELINE, tbl, 0, None, float("inf"), 0.0, 0.0, "FAIL", computed_ts)
        )
        continue

    df = spark.table(tbl)

    # Compute individual metrics
    rc = df.count()
    mx = compute_max_event_ts(df, t["event_ts_col"])
    fm = freshness_minutes(mx)
    nr = compute_null_rate(df, t["null_cols"])
    dr = compute_duplicate_rate(df, t["key_cols"])
    warn_min = float(t.get("warn_min", WARN_MIN))
    fail_min = float(t.get("fail_min", FAIL_MIN))
    st = status_from_freshness(fm, warn_min=warn_min, fail_min=fail_min)

    rows.append(
        (metric_date, PIPELINE, tbl, int(rc), mx, float(fm), float(nr), float(dr), st, computed_ts)
    )

# Define Schema explicitly for safety
schema = T.StructType(
    [
        T.StructField("metric_date", T.DateType(), False),
        T.StructField("pipeline", T.StringType(), False),
        T.StructField("table_name", T.StringType(), False),
        T.StructField("row_count", T.LongType(), False),
        T.StructField("max_event_ts", T.TimestampType(), True),
        T.StructField("freshness_minutes", T.DoubleType(), False),
        T.StructField("null_rate", T.DoubleType(), False),
        T.StructField("duplicate_rate", T.DoubleType(), False),
        T.StructField("status", T.StringType(), False),
        T.StructField("computed_ts", T.TimestampType(), False),
    ]
)

out_df = spark.createDataFrame(rows, schema=schema)
print(f"[INFO] Computation complete. Generated {len(rows)} metrics.")

# 6. Write to Delta & Validate

In [0]:
# 1. Idempotent Write: Delete existing metrics for this date/pipeline first
print(f"[INFO] Overwriting metrics for {metric_date} / {PIPELINE}...")
spark.sql(
    f"DELETE FROM {OBS_TBL} WHERE metric_date = DATE('{metric_date}') AND pipeline = '{PIPELINE}'"
)

# 2. Append new metrics
out_df.write.mode("append").format("delta").saveAsTable(OBS_TBL)
print("[INFO] Observability metrics written successfully.")

# 3. Final Validation Display
display(
    spark.table(OBS_TBL)
    .where((F.col("metric_date") == F.lit(metric_date)) & (F.col("pipeline") == F.lit(PIPELINE)))
    .orderBy(F.col("status").desc(), F.col("table_name"))
)